In [3]:
# Load text from data.txt
from pathlib import Path

candidates = [
    Path.cwd() / "data.txt",
    Path.cwd().parent / "data.txt",
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("data.txt not found in current directory or parent directory.")

text_data = data_path.read_text(encoding="utf-8")
print(f"Loaded {len(text_data)} characters from {data_path}")
print(text_data[:500])

Loaded 50293 characters from c:\projects\learn-rag\chunking_strategies\data.txt
The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th richest sporting league in the world by revenue. It is held annually between March and May. It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1) Fixed-length chunking (no overlap)
fixed_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)
fixed_chunks_no_overlap = fixed_splitter.split_text(text_data)

print("Fixed-length chunks (no overlap):", len(fixed_chunks_no_overlap))
print(fixed_chunks_no_overlap[0][:300])

Fixed-length chunks (no overlap): 228
The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th


In [5]:
# Quick helper to inspect chunking output

def show_chunks(name, chunks, n=3, width=180):
    print(f"\n{name}")
    print("-" * len(name))
    print("Total chunks:", len(chunks))
    for i, chunk in enumerate(chunks[:n], start=1):
        print(f"[{i}] len={len(chunk)} -> {chunk[:width].replace(chr(10), ' ')}")

show_chunks("Fixed-length (no overlap)", fixed_chunks_no_overlap)


Fixed-length (no overlap)
-------------------------
Total chunks: 228
[1] len=300 -> The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it feat
[2] len=299 -> richest sporting league in the world by revenue. It is held annually between March and May. It has an exclusive window in the Future Tours Programme of the International Cricket Co
[3] len=57 -> event, per the Broadcast Audience Research Council.[4][5]


## SpacyTextSplitter - sentence splitting
Sentence-aware chunking using SpaCy sentencizer (no external model download required).

In [6]:
from langchain_text_splitters import SpacyTextSplitter

spacy_splitter = SpacyTextSplitter(
    pipeline="sentencizer",
    chunk_size=500,
    chunk_overlap=75,
)
spacy_chunks = spacy_splitter.split_text(text_data)
show_chunks("SpaCy sentence-aware", spacy_chunks)

Created a chunk of size 877, which is longer than the specified 500
Created a chunk of size 917, which is longer than the specified 500
Created a chunk of size 569, which is longer than the specified 500
Created a chunk of size 535, which is longer than the specified 500
Created a chunk of size 2017, which is longer than the specified 500
Created a chunk of size 1630, which is longer than the specified 500
Created a chunk of size 1068, which is longer than the specified 500
Created a chunk of size 794, which is longer than the specified 500
Created a chunk of size 2105, which is longer than the specified 500
Created a chunk of size 4679, which is longer than the specified 500
Created a chunk of size 1771, which is longer than the specified 500
Created a chunk of size 2464, which is longer than the specified 500
Created a chunk of size 1725, which is longer than the specified 500
Created a chunk of size 1685, which is longer than the specified 500
Created a chunk of size 1127, which is 


SpaCy sentence-aware
--------------------
Total chunks: 69
[1] len=393 -> The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it feat
[2] len=877 -> It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in fewer international tours occurring during the seasons.[3] It is also th
[3] len=129 -> The current champions are the Royal Challengers Bengaluru, who won the 2025 season after defeating the Punjab Kings in the final.


## Semantic text splitting
Splits when semantic similarity between neighboring sentences drops.

In [8]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer


def semantic_split_text(text, max_chunk_chars=1200, similarity_threshold=0.12):
    # Sentence split fallback that works without extra punkt downloads.
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    if len(sentences) <= 1:
        return [text]

    # Local semantic signal using TF-IDF sentence vectors (no remote model download).
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    sent_matrix = vec.fit_transform(sentences).toarray()

    def cosine(a, b):
        denom = (np.linalg.norm(a) * np.linalg.norm(b))
        return float(np.dot(a, b) / denom) if denom else 0.0

    chunks = []
    current = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine(sent_matrix[i - 1], sent_matrix[i])
        proposed = " ".join(current + [sentences[i]])

        # Split on low semantic continuity or if chunk would grow too large.
        if sim < similarity_threshold or len(proposed) > max_chunk_chars:
            chunks.append(" ".join(current))
            current = [sentences[i]]
        else:
            current.append(sentences[i])

    if current:
        chunks.append(" ".join(current))

    return chunks

semantic_chunks = semantic_split_text(text_data)
show_chunks("Semantic splitting", semantic_chunks)


Semantic splitting
------------------
Total chunks: 104
[1] len=349 -> The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it feat
[2] len=42 -> It is held annually between March and May.
[3] len=877 -> It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in fewer international tours occurring during the seasons.[3] It is also th


## Paragraph-based chunking
Splits on blank lines and merges short paragraphs to reduce tiny chunks.

In [9]:

def paragraph_chunk(text, max_chunk_chars=1200):
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks = []
    current = ""

    for p in paras:
        candidate = (current + "\n\n" + p).strip() if current else p
        if len(candidate) <= max_chunk_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = p

    if current:
        chunks.append(current)

    return chunks

paragraph_chunks = paragraph_chunk(text_data)
show_chunks("Paragraph-based", paragraph_chunks)


Paragraph-based
---------------
Total chunks: 45
[1] len=658 -> The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it feat
[2] len=892 -> In 2010, the IPL became the first sporting event to broadcast live on YouTube.[6][7] In 2014, it ranked sixth in attendance among all sports leagues.[8] Inspired by the success of 
[3] len=1041 -> History Winners of the Indian Premier League Season	Winners 2008	Rajasthan Royals 2009	Deccan Chargers 2010	Chennai Super Kings 2011	Chennai Super Kings (2) 2012	Kolkata Knight Rid


## Structure-aware chunking
Uses heading-like lines to preserve section boundaries before splitting.

In [10]:

def structure_aware_chunk(text, max_chunk_chars=1200):
    lines = text.splitlines()
    sections = []
    current = []

    for line in lines:
        stripped = line.strip()
        is_heading = bool(re.match(r"^(#{1,6}\s+|[A-Z][A-Za-z0-9\s]{2,}:$)", stripped))

        if is_heading and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)

    if current:
        sections.append("\n".join(current).strip())

    # Ensure sections are not too large.
    splitter = RecursiveCharacterTextSplitter(chunk_size=max_chunk_chars, chunk_overlap=80)
    chunks = []
    for sec in sections:
        chunks.extend(splitter.split_text(sec))

    return [c for c in chunks if c.strip()]

structure_chunks = structure_aware_chunk(text_data)
show_chunks("Structure-aware", structure_chunks)


Structure-aware
---------------
Total chunks: 58
[1] len=658 -> The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it feat
[2] len=892 -> In 2010, the IPL became the first sporting event to broadcast live on YouTube.[6][7] In 2014, it ranked sixth in attendance among all sports leagues.[8] Inspired by the success of 
[3] len=1041 -> History Winners of the Indian Premier League Season	Winners 2008	Rajasthan Royals 2009	Deccan Chargers 2010	Chennai Super Kings 2011	Chennai Super Kings (2) 2012	Kolkata Knight Rid


## Recursive chunking
Uses ordered separators from coarse to fine for robust general splitting.

In [11]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=450,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""],
)
recursive_chunks = recursive_splitter.split_text(text_data)
show_chunks("Recursive chunking", recursive_chunks)


Recursive chunking
------------------
Total chunks: 174
[1] len=391 -> The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it feat
[2] len=310 -> . It is held annually between March and May. It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in fewer international tours o
[3] len=446 -> In 2010, the IPL became the first sporting event to broadcast live on YouTube.[6][7] In 2014, it ranked sixth in attendance among all sports leagues.[8] Inspired by the success of 


## Query-aware / dynamic chunking
Scores base chunks by query term overlap and expands with neighboring context.

In [12]:

def query_aware_dynamic_chunking(text, query, base_chunk_size=350, top_k=4):
    base_splitter = RecursiveCharacterTextSplitter(chunk_size=base_chunk_size, chunk_overlap=0)
    base_chunks = base_splitter.split_text(text)

    query_terms = set(re.findall(r"\w+", query.lower()))

    scored = []
    for idx, ch in enumerate(base_chunks):
        terms = set(re.findall(r"\w+", ch.lower()))
        score = len(query_terms.intersection(terms))
        scored.append((idx, score))

    ranked_idx = [i for i, _ in sorted(scored, key=lambda x: x[1], reverse=True)[:top_k]]

    dynamic_chunks = []
    for i in sorted(set(ranked_idx)):
        left = base_chunks[i - 1] if i > 0 else ""
        center = base_chunks[i]
        right = base_chunks[i + 1] if i < len(base_chunks) - 1 else ""
        dynamic_chunks.append("\n".join([p for p in [left, center, right] if p]).strip())

    return dynamic_chunks

query = "Who organizes IPL and when is it held?"
query_aware_chunks = query_aware_dynamic_chunking(text_data, query=query)
show_chunks("Query-aware / dynamic", query_aware_chunks)


Query-aware / dynamic
---------------------
Total chunks: 4
[1] len=658 -> The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it feat
[2] len=1005 -> The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it feat
[3] len=851 -> participated from 2011 IPL to 2013 IPL but withdrew due to financial disputes. In 2016, Chennai Super Kings (CSK) and Rajasthan Royals (RR) were suspended for two years due to the 


## Sliding-window chunking with overlap
Creates dense windows to preserve boundary context between adjacent chunks.

In [13]:
sliding_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=150)
sliding_window_chunks = sliding_splitter.split_text(text_data)
show_chunks("Sliding-window with overlap", sliding_window_chunks)


Sliding-window with overlap
---------------------------
Total chunks: 208
[1] len=399 -> The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it feat
[2] len=391 -> richest cricket league in the world and the 11th richest sporting league in the world by revenue. It is held annually between March and May. It has an exclusive window in the Futur
[3] len=160 -> in fewer international tours occurring during the seasons.[3] It is also the most viewed Indian sports event, per the Broadcast Audience Research Council.[4][5]


## Compare all strategies
Summary table to quickly compare chunk counts and average chunk size.

In [14]:
import pandas as pd

all_chunk_sets = {
    "Fixed-length (no overlap)": fixed_chunks_no_overlap,
    "SpaCy sentence-aware": spacy_chunks,
    "Semantic": semantic_chunks,
    "Paragraph-based": paragraph_chunks,
    "Structure-aware": structure_chunks,
    "Recursive": recursive_chunks,
    "Query-aware / dynamic": query_aware_chunks,
    "Sliding-window (overlap)": sliding_window_chunks,
}

summary_rows = []
for name, chunks in all_chunk_sets.items():
    lengths = [len(c) for c in chunks] if chunks else [0]
    summary_rows.append(
        {
            "strategy": name,
            "chunk_count": len(chunks),
            "avg_chunk_len": int(sum(lengths) / len(lengths)),
            "max_chunk_len": max(lengths),
        }
    )

pd.DataFrame(summary_rows).sort_values("strategy").reset_index(drop=True)

,strategy,chunk_count,avg_chunk_len,max_chunk_len
0,Fixed-length (no overlap),228,219,300
1,Paragraph-based,45,1115,5146
2,Query-aware / dynamic,4,763,1005
3,Recursive,174,300,449
4,Semantic,104,482,2461
5,Sliding-window (overlap),208,312,399
6,SpaCy sentence-aware,69,729,4678
7,Structure-aware,58,874,1197
